# Tree diagnostics

Run by the `tree_diagnostics` rule right after `generate_alignment`, once per `{derep}` value.
Parameters come from the JSON file named by `TREE_DIAG_PARAMS`. Tables are also written to the
diagnostics directory so two runs (for example `all` and `derep`) can be compared.

Questions: how complete are the rows of the supermatrix, do genomes of a clade share genes, are
same-clade sequences closer than different-clade ones (overall and gene by gene), how fragmented
are the clades on the tree, and is "take the top-scoring copy" a good rule for multicopy genes?

In [ ]:
import json, os, sys, random
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

P = json.load(open(os.environ["TREE_DIAG_PARAMS"]))
sys.path.insert(0, P["scripts_dir"])
import tree_diagnostics_lib as lib
import tree_fragmentation as frag

OUT = Path(P["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
print({k: v for k, v in P.items() if k != "busco_manifests"})

## 1. Supermatrix occupancy

In [ ]:
ids, matrix = lib.load_alignment(P["alignment"])
partitions = lib.read_partitions(P["partitions"])
taxonomy = lib.read_taxonomy(P["taxonomy"])
occ, present = lib.occupancy(ids, matrix, partitions)
genes_per_genome = np.array(occ["genes"]); gap = np.array(occ["gap_fraction"])
print(f"{len(ids)} genomes x {matrix.shape[1]} columns, {len(partitions)} genes")
print(f"genes per genome: median {np.median(genes_per_genome):.0f}, min {genes_per_genome.min()}, "
      f"10th percentile {np.percentile(genes_per_genome, 10):.0f}")
print(f"gap fraction per row: median {np.median(gap):.2f}, 90th percentile {np.percentile(gap, 90):.2f}, "
      f"rows >= 0.9: {(gap >= 0.9).sum()}")
with open(OUT / "occupancy.tsv", "w") as f:
    f.write("genome\tgenes\tgap_fraction\n")
    for g, n, gp in zip(occ["genome"], occ["genes"], occ["gap_fraction"]):
        f.write(f"{g}\t{n}\t{gp:.4f}\n")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].hist(genes_per_genome, bins=40); ax[0].set_title("genes per genome")
ax[1].hist(gap, bins=40); ax[1].set_title("gap fraction per row")
ax[2].hist(present.sum(axis=0), bins=40); ax[2].set_title("genomes per gene")
plt.show()

Genomes with the fewest genes (candidates to drop or to inspect):

In [ ]:
worst = np.argsort(genes_per_genome)[:15]
for i in worst:
    print(f"{ids[i]:30} genes={genes_per_genome[i]:4d} gaps={gap[i]:.2f} phylum={taxonomy.get(ids[i], {}).get('phylum', '-')}")

## 2. Gene overlap and divergence within vs between clades

Pairs of genomes are sampled; *within* = same clade at the rank, *between* = different clades.
`separation` is the probability that a within-clade pair is closer (p-distance) than a between-clade pair:
1.0 is perfect, 0.5 means the alignment cannot tell the clade from the rest.

In [ ]:
report = lib.rank_report(ids, matrix, present, partitions, taxonomy, n_pairs=P["n_pairs"], seed=1)
rows = [r["summary"] for r in report.values()]
header = list(rows[0]); print("\t".join(header))
for r in rows:
    print("\t".join(f"{r[h]:.3f}" if isinstance(r[h], float) else str(r[h]) for h in header))
with open(OUT / "rank_summary.tsv", "w") as f:
    f.write("\t".join(header) + "\n")
    for r in rows:
        f.write("\t".join(str(r[h]) for h in header) + "\n")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key, title in zip(axes, ["pdist", "jaccard", "shared_genes"], ["p-distance (shared sites)", "gene-set Jaccard", "shared genes"]):
    data, labels, colors = [], [], []
    for rank, r in report.items():
        for kind in ("within", "between"):
            v = np.asarray(r[kind][key], dtype=float); data.append(v[~np.isnan(v)]); labels.append(f"{rank[:4]}\n{kind[:3]}")
    ax.boxplot(data, showfliers=False); ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels, fontsize=7); ax.set_title(title)
plt.show()

### Per-gene separation

Genes whose within-clade distances are not smaller than between-clade ones add noise (saturated,
misaligned or paralogous). Lowest-separation genes per rank:

In [ ]:
gene_rows = {}
for rank in report:
    gs = lib.gene_separation(report, partitions, rank)
    gene_rows[rank] = gs
    good = [g for g in gs if g[2] >= 30 and not np.isnan(g[1])]
    vals = np.array([g[1] for g in good])
    print(f"{rank:8} genes with data={len(good):3d} median separation={np.median(vals):.3f} "
          f"genes below 0.6: {(vals < 0.6).sum()}")
    print("   worst:", ", ".join(f"{g}({s:.2f})" for g, s, _ in sorted(good, key=lambda x: x[1])[:6]))
with open(OUT / "gene_separation.tsv", "w") as f:
    f.write("rank\tgene\tseparation\twithin_pairs_with_data\n")
    for rank, gs in gene_rows.items():
        for g, s, n in gs:
            f.write(f"{rank}\t{g}\t{s}\t{n}\n")
fig, ax = plt.subplots(figsize=(7, 3.5))
for rank, gs in gene_rows.items():
    ax.hist([s for _, s, n in gs if n >= 30 and not np.isnan(s)], bins=25, histtype="step", label=rank)
ax.set_xlabel("per-gene separation"); ax.legend(); plt.show()

## 3. Fragmentation of the clades on the tree

A *piece* is a maximal clade containing only one name. More than one piece means the name is not
monophyletic on this tree. Skipped when no tree has been built yet.

In [ ]:
trees = [t for t in P.get("trees", []) if Path(t).exists()]
if not trees:
    print("no tree available yet; run build_tree and re-run this notebook (or compare trees with scripts/tree_fragmentation.py)")
from Bio import Phylo
for t in trees:
    tree = Phylo.read(t, "newick")
    result = frag.count_pieces(tree, taxonomy)
    print("\n", t, f"({tree.count_terminals()} tips)")
    for r in frag.summarise(result):
        print("  ", r)
    for rank in ("phylum", "class"):
        print(f"   worst {rank}:", ", ".join(f"{n}({tc} tips/{p} pieces)" for p, tc, n in frag.worst(result, rank, 6)))

## 4. Multicopy genes: is "top-scoring copy" a good choice?

For random Duplicated BUSCO genes the copies are compared with the same gene in related genomes
(same family, else order, class, phylum), using 3-mer similarity as a cheap orthology proxy. If the
highest-scoring copy is usually also the most peer-like copy the heuristic is fine; where it is not,
a choice based on similarity to the clade (or the tree itself) could do better.

In [ ]:
tables = []
for entry in P["busco_manifests"]:
    manifest, base = entry["manifest"], Path(entry["base"])
    if not Path(manifest).exists():
        print("missing", manifest); continue
    with open(manifest) as f:
        head = f.readline().rstrip("\n").split("\t"); col = head.index("busco_result")
        for line in f:
            p = line.rstrip("\n").split("\t")[col]
            path = Path(p) if Path(p).is_absolute() else base / p
            tables.append((path.parent.parent.name, str(path)))
print(len(tables), "BUSCO tables")
cases = lib.multicopy_analysis(tables, taxonomy, P["merged_dir"], max_cases=P["max_multicopy_cases"], seed=2) if tables else []
print(len(cases), "evaluated Duplicated genes")

In [ ]:
if cases:
    agree = np.mean([c["agree"] for c in cases])
    gaps = np.array([c["similarity_gap"] for c in cases]); ratio = np.array([c["score_ratio"] for c in cases])
    ident = np.array([c["copy_identity"] for c in cases])
    print(f"top-scoring copy is also the most peer-like copy in {agree:.1%} of cases")
    dis = [c for c in cases if not c["agree"]]
    if dis:
        print(f"when they disagree the better copy is more similar to peers by a median of "
              f"{np.median([c['similarity_gap'] for c in dis]):.3f} (3-mer Jaccard)")
    print(f"median second/first score ratio {np.nanmedian(ratio):.2f}; "
          f"{np.mean(ratio > 0.9):.1%} of genes have a near-tied runner-up (> 0.9)")
    print("peer rank used:", {r: sum(c['peer_rank'] == r for c in cases) for r in ('family', 'order', 'class', 'phylum')})
    with open(OUT / "multicopy_cases.tsv", "w") as f:
        f.write("genome\tgene\tn_copies\tagree\tsimilarity_gap\tscore_ratio\tcopy_identity\tpeer_rank\tn_peers\n")
        for c in cases:
            f.write(f"{c['genome']}\t{c['gene']}\t{c['n_copies']}\t{c['agree']}\t{c['similarity_gap']:.4f}\t"
                    f"{c['score_ratio']:.4f}\t{c['copy_identity']:.4f}\t{c['peer_rank']}\t{c['n_peers']}\n")
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    ax[0].hist(gaps, bins=30); ax[0].set_title("similarity gap (best peer-like copy - chosen copy)")
    ax[1].scatter(ratio, gaps, s=6); ax[1].set_xlabel("score ratio runner-up / top"); ax[1].set_ylabel("gap")
    ax[2].hist(ident, bins=30); ax[2].set_title("similarity between the two best copies")
    plt.show()
else:
    print("no multicopy cases (is use_multi_copy enabled and are there Duplicated genes?)")